# TARGET 4 - Dynamic Route ETA Model

**Goal:** Combine `routes_history` + `weather` + `traffic` + `tir_shipments` (departure_time, route_difficulty) to predict the expected delivery duration (minutes).

**Metric:** MAE (minutes), RMSE (minutes), R2
**Baseline:** simple `distance_km / avg_speed` ETA
**Models compared:** 6 different regressors, best one is tuned with full hyperparameter search
**Final layer:** LLM-generated natural-language ETA brief for dispatcher + customer

| File | Columns | Role |
|---|---|---|
| routes_history.parquet | distance_km, duration_minutes, fuel_used | Historical base ETA + TARGET |
| tir_shipments.parquet | departure_time, route_difficulty, actual_load_ton | Trip conditions |
| weather.parquet | rainfall_mm, wind_speed | Weather delay factor |
| traffic.parquet | congestion_level | Road conditions |


## 1. Libraries and data loading

In [26]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
import joblib
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


routes  = pd.read_parquet('routes_history.parquet')
tir     = pd.read_parquet('tir_shipments.parquet')
traffic = pd.read_parquet('traffic.parquet')
weather = pd.read_parquet('weather.parquet')

print('routes_history :', routes.shape)
print('tir_shipments  :', tir.shape)
print('traffic        :', traffic.shape)
print('weather        :', weather.shape)


routes_history : (3000, 7)
tir_shipments  : (16000, 17)
traffic        : (30000, 3)
weather        : (23570, 5)


In [27]:

routes.head(3)


,route_id,vehicle_id,start_location,end_location,distance_km,duration_minutes,fuel_used
0,RT30000,VH1081,Qazakh,Khachmaz,681.9,653.6,142.07
1,RT30001,VH1031,Lankaran,Ganja,726.0,701.0,79.32
2,RT30002,VH1060,Yevlakh,Lankaran,791.4,982.8,67.71


In [28]:

tir.head(3)


,shipment_id,route_id,origin_hub,destination_hub,departure_date,departure_time,actual_load_ton,capacity_ton,utilization_rate,num_packages,is_delayed,delay_minutes,is_spot_rental,driver_id,route_difficulty,associated_order_ids,pricing_confidence
0,SH40000,RT32459,HUB_SHEKI,HUB_KHANKENDI,2025-01-16,23:30:00,12.506,13.84,0.903613,9,0,0.0,0,DR648,flat,OR946151|OR930661|OR902341|OR946491|OR901044,market_estimate
1,SH40001,RT30672,HUB_NAKHCHIVAN,HUB_KHACHMAZ,2025-09-14,22:30:00,11.944,12.74,0.937520,10,0,0.0,0,DR573,flat,OR925651|OR923180|OR935502|OR935463|OR929429,market_estimate
2,SH40002,RT31922,HUB_KHANKENDI,HUB_KALBAJAR,2021-10-31,22:30:00,15.225,16.33,0.932333,9,0,0.0,1,DR612,flat,OR933071|OR932503,market_estimate


## 2. Schema check

`tir_shipments.route_id` matches `routes_history.route_id` (TARGET = `duration_minutes`).
`origin_hub` (`HUB_XXX`) needs to be mapped to `weather.region` / `traffic.region` (`XXX`) by stripping the prefix.


In [29]:

print('route_id overlap (tir -> routes):', tir['route_id'].isin(routes['route_id']).mean().round(4))
print('region names (weather):', sorted(weather['region'].unique()))
print('hub names (tir):', sorted(tir['origin_hub'].unique()))


route_id overlap (tir -> routes): 0.7484
region names (weather): ['Absheron', 'Ganja', 'Kalbajar', 'Khachmaz', 'Khankendi', 'Lankaran', 'Nakhchivan', 'Qazakh', 'Sheki', 'Yevlakh']
hub names (tir): ['HUB_ABSHERON', 'HUB_GANJA', 'HUB_KALBAJAR', 'HUB_KHACHMAZ', 'HUB_KHANKENDI', 'HUB_LANKARAN', 'HUB_NAKHCHIVAN', 'HUB_QAZAKH', 'HUB_SHEKI', 'HUB_YEVLAKH']


## 3. Building the dataset (merge)

**3.1 — `tir_shipments` + `routes_history`** (inner join on `route_id`) → adds the TARGET (`duration_minutes`).

In [30]:

df = tir.merge(
    routes[['route_id', 'start_location', 'end_location', 'distance_km', 'duration_minutes', 'fuel_used']],
    on='route_id', how='inner'
)
df = df.reset_index(drop=True)
print('Merged shipment count:', df.shape)
df.head(3)


Merged shipment count: (11975, 22)


,shipment_id,route_id,origin_hub,destination_hub,departure_date,departure_time,actual_load_ton,capacity_ton,utilization_rate,num_packages,is_delayed,delay_minutes,is_spot_rental,driver_id,route_difficulty,associated_order_ids,pricing_confidence,start_location,end_location,distance_km,duration_minutes,fuel_used
0,SH40000,RT32459,HUB_SHEKI,HUB_KHANKENDI,2025-01-16,23:30:00,12.506,13.84,0.903613,9,0,0.0,0,DR648,flat,OR946151|OR930661|OR902341|OR946491|OR901044,market_estimate,Nakhchivan,Sheki,583.9,605.9,110.13
1,SH40001,RT30672,HUB_NAKHCHIVAN,HUB_KHACHMAZ,2025-09-14,22:30:00,11.944,12.74,0.937520,10,0,0.0,0,DR573,flat,OR925651|OR923180|OR935502|OR935463|OR929429,market_estimate,Sheki,Absheron,700.8,745.3,123.96
2,SH40002,RT31922,HUB_KHANKENDI,HUB_KALBAJAR,2021-10-31,22:30:00,15.225,16.33,0.932333,9,0,0.0,1,DR612,flat,OR933071|OR932503,market_estimate,Khachmaz,Ganja,651.1,753.1,87.25


**3.2 — Building the departure timestamp and converting hub names to regions**

In [31]:

df['departure_dt'] = pd.to_datetime(df['departure_date'] + ' ' + df['departure_time'])
df['region'] = df['origin_hub'].str.replace('HUB_', '', regex=False).str.title()

# handle any naming mismatches between hub names and region names
region_map = {r.title(): r for r in weather['region'].unique()}
df['region'] = df['region'].map(lambda r: region_map.get(r, r))

print(df['region'].value_counts())


region
Lankaran      1247
Qazakh        1243
Khankendi     1227
Kalbajar      1224
Nakhchivan    1216
Ganja         1205
Yevlakh       1203
Khachmaz      1174
Absheron      1120
Sheki         1116
Name: count, dtype: int64


**3.3 — Joining weather data** (`weather` is daily → join by region + nearest date using `merge_asof`)

In [32]:

weather_sorted = weather.copy()
weather_sorted['timestamp'] = pd.to_datetime(weather_sorted['timestamp']).dt.tz_localize(None)
weather_sorted = weather_sorted.sort_values('timestamp')

df = df.sort_values('departure_dt')

merged_parts = []
for region, grp in df.groupby('region'):
    w_region = weather_sorted[weather_sorted['region'] == region].sort_values('timestamp')
    grp = grp.sort_values('departure_dt')
    orig_index = grp.index
    if w_region.empty:
        grp = grp.copy()
        grp['rainfall'] = np.nan
        grp['wind_speed'] = np.nan
        grp['temperature'] = np.nan
    else:
        grp = pd.merge_asof(
            grp.reset_index(drop=True), w_region[['timestamp', 'temperature', 'rainfall', 'wind_speed']],
            left_on='departure_dt', right_on='timestamp', direction='backward'
        ).drop(columns='timestamp')
        grp.index = orig_index
    merged_parts.append(grp)

df = pd.concat(merged_parts).sort_index()
print('After joining weather:', df.shape)
df[['region', 'departure_dt', 'rainfall', 'wind_speed', 'temperature']].head(3)


After joining weather: (11975, 27)


,region,departure_dt,rainfall,wind_speed,temperature
0,Sheki,2025-01-16 23:30:00,0.0,14.128566,6.945
1,Nakhchivan,2025-09-14 22:30:00,0.0,24.131413,27.850
2,Khankendi,2021-10-31 22:30:00,0.0,9.220499,17.172


**3.4 — Joining traffic data** (region + nearest timestamp using `merge_asof`)

In [33]:

traffic_sorted = traffic.copy()
traffic_sorted['timestamp'] = pd.to_datetime(traffic_sorted['timestamp'])

merged_parts = []
for region, grp in df.groupby('region'):
    t_region = traffic_sorted[traffic_sorted['region'] == region].sort_values('timestamp')
    grp = grp.sort_values('departure_dt')
    orig_index = grp.index
    if t_region.empty:
        grp = grp.copy()
        grp['traffic_congestion_level'] = np.nan
    else:
        grp = pd.merge_asof(
            grp.reset_index(drop=True), t_region[['timestamp', 'traffic_congestion_level']],
            left_on='departure_dt', right_on='timestamp', direction='backward'
        ).drop(columns='timestamp')
        grp.index = orig_index
    merged_parts.append(grp)

df = pd.concat(merged_parts).sort_index()
print('After joining traffic:', df.shape)
df[['region', 'departure_dt', 'traffic_congestion_level']].head(3)


After joining traffic: (11975, 28)


,region,departure_dt,traffic_congestion_level
0,Sheki,2025-01-16 23:30:00,1.7087
1,Nakhchivan,2025-09-14 22:30:00,3.1952
2,Khankendi,2021-10-31 22:30:00,1.1464


In [34]:

print('Missing (NaN) values:')
print(df[['rainfall', 'wind_speed', 'temperature', 'traffic_congestion_level']].isna().sum())

# fill any remaining NaNs with the region median
for c in ['rainfall', 'wind_speed', 'temperature', 'traffic_congestion_level']:
    df[c] = df.groupby('region')[c].transform(lambda s: s.fillna(s.median()))
    df[c] = df[c].fillna(df[c].median())


Missing (NaN) values:
rainfall                    0
wind_speed                  0
temperature                 0
traffic_congestion_level    3
dtype: int64


## 4. Feature engineering

In [35]:

df['departure_hour'] = df['departure_dt'].dt.hour
df['day_of_week'] = df['departure_dt'].dt.dayofweek          # 0 = Monday
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_night'] = df['departure_hour'].isin(list(range(22, 24)) + list(range(0, 6))).astype(int)

# route_difficulty one-hot (flat / mountainous)
df['is_mountainous'] = (df['route_difficulty'] == 'mountainous').astype(int)

feature_cols = [
    'distance_km',
    'is_mountainous',
    'departure_hour',
    'day_of_week',
    'is_weekend',
    'is_night',
    'rainfall',
    'wind_speed',
    'temperature',
    'traffic_congestion_level',
    'actual_load_ton',
    'capacity_ton',
    'utilization_rate',
    'num_packages',
    'is_spot_rental',
]

target_col = 'duration_minutes'

extra_cols = ['shipment_id', 'route_id', 'start_location', 'end_location', 'departure_dt']
model_cols = feature_cols + [target_col] + [c for c in extra_cols if c not in feature_cols]
model_df = df[model_cols].dropna()
print('Final dataset for modeling:', model_df.shape)
model_df[feature_cols + [target_col]].describe().T


Final dataset for modeling: (11975, 21)


,count,mean,std,min,25%,50%,75%,max
distance_km,11975.0,430.204752,214.788402,55.200000,244.100000,426.300000,616.850000,799.700000
is_mountainous,11975.0,0.299040,0.457856,0.000000,0.000000,0.000000,1.000000,1.000000
departure_hour,11975.0,10.144134,10.214099,0.000000,2.000000,3.000000,22.000000,23.000000
day_of_week,11975.0,2.980960,1.999199,0.000000,1.000000,3.000000,5.000000,6.000000
is_weekend,11975.0,0.285428,0.451637,0.000000,0.000000,0.000000,1.000000,1.000000
is_night,11975.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
rainfall,11975.0,1.980443,5.779898,0.000000,0.000000,0.000000,1.000000,144.499980
wind_speed,11975.0,13.826669,7.152888,2.052316,8.714677,11.726277,16.923828,58.351105
temperature,11975.0,18.243281,9.693565,-7.948000,10.400000,17.950000,25.854750,41.950000
traffic_congestion_level,11975.0,1.844481,1.162920,0.200000,0.949900,1.598300,2.469300,5.000000


## 5. Train / Test split — by TIME, not randomly

**Fix (this version):** the split was previously a random `train_test_split(test_size=0.2)`, which is inconsistent with Targets 1-3 (all of which use a chronological `TimeSeriesSplit` / time-based cutoff) and carries a theoretical leakage risk — a shipment from a given day could land in train while another shipment from the *same day* (sharing the same day's weather/traffic snapshot) lands in test, letting same-day information leak across the split.

The split below uses the same convention as Target 3: sort by `departure_dt`, cut at the 85th percentile (train = earliest ~85% of shipments, test = the most recent ~15%). This guarantees every test-set shipment departs **strictly after** every train-set shipment, matching how the model would actually be used in production (trained on the past, evaluated on the future).

In [36]:
model_df = model_df.sort_values('departure_dt').reset_index(drop=True)

cutoff_t4 = model_df['departure_dt'].quantile(0.85)
train_mask_t4 = model_df['departure_dt'] < cutoff_t4

X = model_df[feature_cols]
y = model_df[target_col]
meta_cols = ['shipment_id', 'route_id', 'start_location', 'end_location', 'departure_dt', 'distance_km']

X_train, X_test = X[train_mask_t4], X[~train_mask_t4]
y_train, y_test = y[train_mask_t4], y[~train_mask_t4]
meta_train = model_df.loc[train_mask_t4, meta_cols]
meta_test  = model_df.loc[~train_mask_t4, meta_cols]

print(f'Cutoff date: {cutoff_t4.date()}')
print('Train:', X_train.shape, ' Test:', X_test.shape)
print(f'Train period: {meta_train["departure_dt"].min().date()} -> {meta_train["departure_dt"].max().date()}')
print(f'Test period:  {meta_test["departure_dt"].min().date()} -> {meta_test["departure_dt"].max().date()}')

Cutoff date: 2025-06-27
Train: (10178, 15)  Test: (1797, 15)
Train period: 2020-01-01 -> 2025-06-27
Test period:  2025-06-27 -> 2026-06-12


**Leakage verification test** — every test-set shipment must depart strictly after every train-set shipment. This is a hard requirement of a chronological split; the assertion below proves it rather than assuming it.

In [37]:
assert meta_train['departure_dt'].max() < meta_test['departure_dt'].min(), \
    "LEAKAGE: a train-set shipment departs after a test-set shipment!"

print("LEAKAGE TEST: PASS")
print(f"  Last train departure : {meta_train['departure_dt'].max()}")
print(f"  First test departure : {meta_test['departure_dt'].min()}")
print(f"  Gap: {(meta_test['departure_dt'].min() - meta_train['departure_dt'].max())}")

LEAKAGE TEST: PASS
  Last train departure : 2025-06-27 00:45:00
  First test departure : 2025-06-27 02:45:00
  Gap: 0 days 02:00:00


## 6. XGBoost — Default Parameters


In [38]:
from xgboost import XGBRegressor

xgb_default = XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror')
xgb_default.fit(X_train, y_train)

pred_train = xgb_default.predict(X_train)
pred_test  = xgb_default.predict(X_test)

print("Model: XGBoost Regressor (default parameters)")
print(f"Train -> RMSE={mean_squared_error(y_train, pred_train)**0.5:.2f}  MAE={mean_absolute_error(y_train, pred_train):.2f}  R2={r2_score(y_train, pred_train):.4f}")
print(f"Test  -> RMSE={mean_squared_error(y_test, pred_test)**0.5:.2f}  MAE={mean_absolute_error(y_test, pred_test):.2f}  R2={r2_score(y_test, pred_test):.4f}")


Model: XGBoost Regressor (default parameters)
Train -> RMSE=28.71  MAE=21.95  R2=0.9850
Test  -> RMSE=58.13  MAE=44.03  R2=0.9379


### XGBoost — Hyperparameter Tuning (GridSearchCV)


In [39]:
from sklearn.model_selection import GridSearchCV
import joblib

xgb_param_grid = {
    'n_estimators':     [300],
    'learning_rate':    [0.02],
    'max_depth':        [5],
    'subsample':        [0.9],
    'colsample_bytree': [1],
    'min_child_weight': [1],
    'reg_alpha':        [5],
    'reg_lambda':       [7],
}
CV_FOLDS = 10

grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=1, objective='reg:squarederror'),
    param_grid=xgb_param_grid,
    scoring='r2',
    cv=CV_FOLDS,
    n_jobs=4,
    pre_dispatch='2*n_jobs',
    verbose=1,
)

with joblib.parallel_backend('threading'):
    grid_search.fit(X_train, y_train)

xgb_model = grid_search.best_estimator_
pred_train = xgb_model.predict(X_train)
pred_test  = xgb_model.predict(X_test)

r2_train = r2_score(y_train, pred_train)
r2_test  = r2_score(y_test, pred_test)

print(f"Best params: {grid_search.best_params_}")
print(f"Train -> RMSE={mean_squared_error(y_train, pred_train)**0.5:.2f}  MAE={mean_absolute_error(y_train, pred_train):.2f}  R2={r2_train:.4f}")
print(f"Test  -> RMSE={mean_squared_error(y_test, pred_test)**0.5:.2f}  MAE={mean_absolute_error(y_test, pred_test):.2f}  R2={r2_test:.4f}")
print(f"Train/Test R2 gap: {r2_train - r2_test:.4f}")

Fitting 10 folds for each of 1 candidates, totalling 10 fits
Best params: {'colsample_bytree': 1, 'learning_rate': 0.02, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 300, 'reg_alpha': 5, 'reg_lambda': 7, 'subsample': 0.9}
Train -> RMSE=49.95  MAE=38.50  R2=0.9546
Test  -> RMSE=53.87  MAE=41.58  R2=0.9467
Train/Test R2 gap: 0.0079


## 7. Save Best (Tuned) Model with joblib


In [40]:
import joblib, os

os.makedirs("models", exist_ok=True)
model_path = "models/target4_XGBoost_tuned.joblib"
joblib.dump(xgb_model, model_path)

print(f"Saved tuned model to: {model_path}")


Saved tuned model to: models/target4_XGBoost_tuned.joblib


## 8. 7-Day ETA Forecast (by Route)

There is no calendar-indexed history per route (each route recurs with different departure times), so the forecast is built per (route x future day) using each route's historical average conditions.


In [41]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

FORECAST_DAYS = 7
last_date_t4 = df['departure_dt'].max().normalize()
future_dates_t4 = pd.date_range(last_date_t4 + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq='D')

route_profile = model_df.groupby('route_id').agg({
    'distance_km': 'mean',
    'is_mountainous': 'first',
    'rainfall': 'mean',
    'wind_speed': 'mean',
    'temperature': 'mean',
    'traffic_congestion_level': 'mean',
    'actual_load_ton': 'mean',
    'capacity_ton': 'mean',
    'utilization_rate': 'mean',
    'num_packages': 'mean',
    'is_spot_rental': lambda x: x.mode().iloc[0] if len(x.mode()) else 0,
    'departure_hour': lambda x: int(x.mode().iloc[0]) if len(x.mode()) else 12,
    'start_location': 'first',
    'end_location': 'first',
}).reset_index()

# Build ALL (route x day) rows first, then predict once in a single batch call
rows = []
meta = []
for _, route in route_profile.iterrows():
    is_night = int(route['departure_hour'] in list(range(22, 24)) + list(range(0, 6)))
    for fdate in future_dates_t4:
        rows.append({
            'distance_km': route['distance_km'],
            'is_mountainous': route['is_mountainous'],
            'departure_hour': route['departure_hour'],
            'day_of_week': fdate.dayofweek,
            'is_weekend': int(fdate.dayofweek >= 5),
            'is_night': is_night,
            'rainfall': route['rainfall'],
            'wind_speed': route['wind_speed'],
            'temperature': route['temperature'],
            'traffic_congestion_level': route['traffic_congestion_level'],
            'actual_load_ton': route['actual_load_ton'],
            'capacity_ton': route['capacity_ton'],
            'utilization_rate': route['utilization_rate'],
            'num_packages': route['num_packages'],
            'is_spot_rental': route['is_spot_rental'],
        })
        meta.append({
            'route_id': route['route_id'],
            'start_location': route['start_location'],
            'end_location': route['end_location'],
            'date': fdate.date(),
            'day_of_week': fdate.day_name(),
        })

X_batch = pd.DataFrame(rows)[feature_cols]
preds = xgb_model.predict(X_batch)              # ONE call for everything, no loop-spam
preds = [round(float(max(0.0, p)), 1) for p in preds]

forecast_df_t4 = pd.DataFrame(meta)
forecast_df_t4['predicted_duration_minutes'] = preds

pivot_t4 = forecast_df_t4.pivot_table(
    index=['route_id', 'start_location', 'end_location'],
    columns='date', values='predicted_duration_minutes'
)

print(f"7-day ETA outlook ({future_dates_t4[0].date()} to {future_dates_t4[-1].date()}), model: XGBoost (tuned)")
pivot_t4


7-day ETA outlook (2026-06-13 to 2026-06-19), model: XGBoost (tuned)


,,date,2026-06-13,2026-06-14,2026-06-15,2026-06-16,2026-06-17,2026-06-18,2026-06-19
route_id,start_location,end_location,,,,,,,
RT30000,Qazakh,Khachmaz,725.2,724.1,725.9,725.9,725.5,724.2,724.9
RT30001,Lankaran,Ganja,777.6,780.7,772.0,772.0,772.5,773.1,777.3
RT30002,Yevlakh,Lankaran,875.4,882.1,869.0,869.6,871.9,872.2,875.2
RT30003,Khankendi,Lankaran,147.0,147.0,147.0,147.0,147.0,147.0,147.0
RT30004,Sheki,Khankendi,191.9,192.4,191.8,191.8,191.9,191.9,191.9
...,...,...,...,...,...,...,...,...,...
RT32994,Khankendi,Nakhchivan,500.3,501.1,501.4,501.4,500.6,500.3,500.3
RT32995,Lankaran,Absheron,594.3,595.3,600.3,596.5,595.3,594.3,594.3
RT32996,Nakhchivan,Yevlakh,500.6,501.5,501.7,501.7,500.9,500.6,500.6


### 7-day ETA forecast — JSON output


In [42]:
import json

json_records_t4 = []
for route_id, grp in forecast_df_t4.groupby('route_id'):
    grp = grp.sort_values('date')
    first = grp.iloc[0]
    json_records_t4.append({
        "route_id": route_id,
        "start_location": first['start_location'],
        "end_location": first['end_location'],
        "model": "XGBoost_tuned",
        "forecast": [
            {"date": str(r['date']), "day_of_week": r['day_of_week'], "predicted_duration_minutes": float(r['predicted_duration_minutes'])}
            for _, r in grp.iterrows()
        ],
    })

forecast_json_t4 = json.dumps(json_records_t4, indent=2, ensure_ascii=False)

with open("target4_7day_forecast.json", "w", encoding="utf-8") as f:
    f.write(forecast_json_t4)

print(f"Saved 7-day ETA forecast for {len(json_records_t4)} routes to target4_7day_forecast.json")
print(forecast_json_t4[:800])


Saved 7-day ETA forecast for 2943 routes to target4_7day_forecast.json
[
  {
    "route_id": "RT30000",
    "start_location": "Qazakh",
    "end_location": "Khachmaz",
    "model": "XGBoost_tuned",
    "forecast": [
      {
        "date": "2026-06-13",
        "day_of_week": "Saturday",
        "predicted_duration_minutes": 725.2
      },
      {
        "date": "2026-06-14",
        "day_of_week": "Sunday",
        "predicted_duration_minutes": 724.1
      },
      {
        "date": "2026-06-15",
        "day_of_week": "Monday",
        "predicted_duration_minutes": 725.9
      },
      {
        "date": "2026-06-16",
        "day_of_week": "Tuesday",
        "predicted_duration_minutes": 725.9
      },
      {
        "date": "2026-06-17",
        "day_of_week": "Wednesday",
        "predicted_duration_minutes": 725.5
      },
      {
        "date": "2026
